In [172]:
import numpy as np
import random
from collections import Counter

In [173]:
data_path = "../data/raw/text8"
with open(data_path, 'r') as f:
    raw_data = f.read().split()

In [174]:
for i in range(0, 10):
    print(raw_data[i])

anarchism
originated
as
a
term
of
abuse
first
used
against


In [175]:
len(raw_data)

17005207

In [206]:
small_raw_data = raw_data[0:10000]

In [177]:
def create_vocab(words, min_count = 5):
    counts = Counter(words)

    vocab = [word for word, freq in counts.items() if freq >= min_count]

    word_to_id = {word: i for i, word in enumerate(vocab)}

    id_to_word = {i: word for word, i in word_to_id.items()}

    return word_to_id, id_to_word

In [178]:
def words_to_ids(words, word_to_id):
    return [word_to_id[w] for w in words if w in word_to_id]

In [199]:
class SkipgramModel:
    def __init__(self, vocab_size, embed_size, learning_rate = 0.1):
        self.v_size = vocab_size
        self.e_size = embed_size
        self.lr = learning_rate

        self.W1 = np.random.randn(self.v_size, self.e_size) * 0.1
        self.W2 = np.random.randn(self.e_size, self.v_size) * 0.1

    def _softmax(self, x):
        e_x = np.exp(x - np.max(x))
        
        return e_x / e_x.sum(axis = 0)

    def train_step(self, center_word_idx, context_word_indices):
        h = self.W1[center_word_idx]

        u = np.dot(h, self.W2)
        y_pred = self._softmax(u)

        total_error = np.zeros(self.v_size)
        for target_idx in context_word_indices:
            e = y_pred.copy()
            e[target_idx] -= 1
            total_error += e

        dW2 = np.outer(h, total_error)
        dW1 = np.dot(self.W2, total_error)

        self.W2 -= self.lr * dW2
        self.W1[center_word_idx] -= self.lr * dW1

        loss = -np.mean([np.log(y_pred[idx] + 1e-9) for idx in context_word_indices])

        return loss

In [192]:
word_to_id, id_to_word = create_vocab(small_raw_data, min_count = 5)
data_as_ids = words_to_ids(small_raw_data, word_to_id)

In [203]:
VOCAB_SIZE = len(word_to_id)
EMBED_SIZE = 100
WINDOW_SIZE = 2
BATCH_SIZE = 128
LEARNING_RATE = 0.01
EPOCHS = 10

In [204]:
model = SkipgramModel(VOCAB_SIZE, EMBED_SIZE, LEARNING_RATE)

In [207]:
for epoch in range(EPOCHS):
    total_loss = 0
    steps = 0

    for i in range(len(data_as_ids)):
        center_id = data_as_ids[i]

        start = max(0, i - window_size)
        end = min(len(data_as_ids), i + window_size + 1)

        context_ids = [data_as_ids[j] for j in range(start, end) if i != j]

        steps += 1
        loss = model.train_step(center_id, context_ids)
        total_loss += loss

    print(f"Epoch {epoch} finished, avg. loss: {total_loss/steps}")
        

Epoch 0 finished, avg. loss: 4.5680698898743
Epoch 1 finished, avg. loss: 4.550991364353393
Epoch 2 finished, avg. loss: 4.535220508390843
Epoch 3 finished, avg. loss: 4.52069005688019
Epoch 4 finished, avg. loss: 4.507325022099896
Epoch 5 finished, avg. loss: 4.4950338192701995
Epoch 6 finished, avg. loss: 4.483712320231476
Epoch 7 finished, avg. loss: 4.473256715733291
Epoch 8 finished, avg. loss: 4.4635738524655
Epoch 9 finished, avg. loss: 4.454584283540712
